In [2]:
from ref import *
from read_complex_matrix import read_complex_matrix
from calculating_dynamic_resolution import *
import pandas as pd
import torch
import torch.optim as optim
torch.set_printoptions(precision=10)

matr = read_complex_matrix('ref_matrix.txt')
import timeit


import warnings

warnings.filterwarnings("ignore", category=UserWarning, message="To copy construct from a tensor")

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [4]:
q = dynamic_mesh_q(kmax, Ndots, gap)

r, r_conv= reflectometry(q, matr)

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [ ]:
r.device

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

ax.plot(q.detach().numpy()[0:len(r)]*1e-9, r)
ax.plot(q.detach().numpy()[0:len(r_conv)]*1e-9, r_conv)
#ax.scatter(x_max.detach().numpy()*1e-9, y_max.detach().numpy(), color = 'orange')
#ax.scatter(x_min.detach().numpy()*1e-9, y_min.detach().numpy(), color = 'cyan')
#ax.scatter(x_crit.detach().numpy()*1e-9, 0.9, color = 'red')

ax.set_yscale('log')

plt.show()

In [ ]:
class Comparator:
    def __init__(self, q, orig_matr):
        self.q = q
        self.original_ref = reflectometry(q, orig_matr)
        
    def compare_trace(self, var_matr):
        r1, r_conv1 = self.original_ref
        r2, r_conv2 = reflectometry(self.q, var_matr)
        #print(var_matr)
        
        #return (torch.sum(((r_conv1-r_conv2)/r_conv1)**2)) #
        return(1e+6*torch.sum(((r1-r2)/r1)**2))

# Вызываем compare много раз без повторной загрузки reference_points
loss_function = Comparator(q, matr)

In [ ]:
all_bounds = torch.tensor([[[1.0000e-09, 1.0000e-09], [0.0000e+00, 0.0000e+00],  [0.0000e+00, 0.0000e+00]],
                           [[1.0000e-08, 9.0000e-08], [8.0000e+14, 8.0000e+14],  [3.0000e-09, 5.0000e-09]],
                           [[1.0000e-08, 3.0000e-08], [-2.0000e+14, -2.0000e+14],[1.0000e-09, 9.0000e-09]],
                           [[1.0000e-09, 1.0000e-09], [2.0000e+14, 2.0000e+14],  [0.0000e+00, 0.0000e+00]]])

In [ ]:
class varbounds:
    def __init__(self, bounds_tensor):
        self.bounds = torch.tensor(bounds_tensor).clone()

    def objective_function(self, var_vector):
        matrixx = self.bounds[..., 0].clone()
        matrixx[torch.where(self.bounds[..., 0] != self.bounds[..., 1])] =  torch.tensor(var_vector*1e-9).clone()
        return loss_function.compare_trace(matrixx)

objective_function = varbounds(all_bounds)

In [ ]:
#в нм
print(objective_function.objective_function(torch.tensor([40, 3, 20, 3])))

In [ ]:
initial_guess = torch.tensor([40.0, 3.0, 15.0, 2.0]).requires_grad_(True)

In [ ]:
def tryfun(vector):
    a = vector[0]
    b = vector[1]
    c = vector[2]
    d = vector[3]
    return (a-1)**2 + (b-2)**2 + (c-3)**2 + (d-4)**2

In [ ]:
def numerical_gradient(fun, x, eps=1e-4):
    x = x.detach().clone()  # Отключаем автоград и делаем независимую копию
    grad = torch.zeros_like(x)
    
    # Сохраняем исходную форму
    original_shape = x.shape
    x_flat = x.view(-1)
    grad_flat = grad.view(-1)
    
    for i in range(x_flat.numel()):
        # Создаем копии для каждого направления
        x_pos = x_flat.clone()
        x_neg = x_flat.clone()
        
        # Модифицируем конкретный элемент
        x_pos[i] = x_flat[i] + eps
        x_neg[i] = x_flat[i] - eps
        
        # Возвращаем исходную форму для вычисления функции
        f_pos = fun(x_pos.view(original_shape))
        f_neg = fun(x_neg.view(original_shape))
        
        # Вычисляем градиент
        diff = f_pos - f_neg
        grad_flat[i] = diff / (2 * eps)

    return grad.view(original_shape)

In [ ]:
def adamw_useless(fun, x0, lr=3e-11, max_iter=16000, tol=2e-12):
    x = x0.clone().detach().requires_grad_(True)
    optimizer = torch.optim.AdamW([x], lr=lr)
    
    prev_loss = None
    
    for i in range(max_iter):
        optimizer.zero_grad()
        
        loss = fun(x)
        grad = numerical_gradient(fun, x.detach())
        x.grad = grad
        optimizer.step()
        if i%50 == 0: print(loss, "   ",grad,"  ",  x,  "\n")
        
        loss_val = loss.item()
        if prev_loss is not None and abs(prev_loss - loss_val) < tol and i>=600:
            print(f"Converged at iteration {i}, loss: {loss_val}")
            break
        prev_loss = loss_val
    
    return x.detach(), prev_loss

In [ ]:
def numerical_gradient(func, x, epsilon=1e-3):
    grad = torch.zeros_like(x)
    x_flat = x.view(-1)
    grad_flat = grad.view(-1)
    
    for i in range(x_flat.size(0)):
        original = x_flat[i].item()
        
        # Создаем копии для возмущений
        x_plus = x.detach().clone()
        x_plus.view(-1)[i] = original + epsilon
        f_plus = func(x_plus)
        
        x_minus = x.detach().clone()
        x_minus.view(-1)[i] = original - epsilon
        f_minus = func(x_minus)
        
        # Центральная разностная производная
        grad_flat[i] = (f_plus - f_minus) / (2 * epsilon)
    
    return grad

def adamw(func, initial_point, lr=3e-2, max_iter=10000, tol=1e-10, epsilon=1e-6):
    x = torch.nn.Parameter(initial_point.clone().detach(), requires_grad=False)
    optimizer = optim.AdamW([x], lr=lr, weight_decay=0)  # Убрали weight decay
    prev_loss = None
    
    for i in range(max_iter):
        optimizer.zero_grad()
        
        current_loss = func(x)
        grad = numerical_gradient(func, x, epsilon)  # Убедитесь, что функция корректна
        x.grad = grad.clone()
        
        if i % 60 == 0:
            print(f"Iteration {i}: Loss: {current_loss.item():.8f}")
        
        optimizer.step()
        
        # Ранняя остановка при малом изменении потерь
        if prev_loss is not None and abs(prev_loss - current_loss) < tol and i >= 50:
            print(f"Converged at iteration {i}")
            break
            
        prev_loss = current_loss.item()
    
    return x.detach(), current_loss.item()

In [ ]:
%time
ans, loss = adamw(objective_function.objective_function, initial_guess)
print(ans)
print(loss)

In [ ]:
initial_guess

In [ ]:
#не работает. Сходится абы куда.

def sgd(fun, x0, lr=2e-3, max_iter=16000, tol=2e-12):
    x = x0.clone().detach().requires_grad_(True)
    optimizer = torch.optim.SGD([x], lr=lr)
    
    prev_loss = None
    
    for i in range(max_iter):
        optimizer.zero_grad()
        
        loss = fun(x)
        grad = numerical_gradient(fun, x.detach())
        x.grad = grad
        optimizer.step()
        if i%50 == 0: print(loss, "   ",grad,"  ",  x,  "\n")
        
        loss_val = loss.item()
        if prev_loss is not None and abs(prev_loss - loss_val) < tol and i>=600:
            print(f"Converged at iteration {i}, loss: {loss_val}")
            break
        prev_loss = loss_val
    
    return x.detach(), prev_loss


In [ ]:
"""
ans, loss = sgd(objective_function.objective_function, initial_guess)
print(ans)
print(loss)
"""